In [1]:
"""
Merpis API (year=all) -> 1 baris per project (T1 ringkas + T2 semua aktivitas)
-> upsert ke Supabase.  Default DRY_RUN=true (tidak menulis apa pun).
"""
import os, re, json, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests

try:
    from dotenv import load_dotenv
    load_dotenv("environment.env", override=True)
except ImportError:
    pass

# ======================= KONFIGURASI =======================
MERPIS_URL   = os.environ["MERPIS_API_URL"]
MERPIS_TOKEN = os.environ["MERPIS_API_TOKEN"]          # sudah berisi "Bearer ..."
SB_URL       = os.environ["SUPABASE_URL"].rstrip("/")
SB_KEY       = os.environ["SUPABASE_KEY"]
TABLE        = os.environ.get("TURNAROUND_TABLE", "monitoring_turnaround_v2")
KEY          = "project_code"
BATCH        = 200
MIN_RATIO    = 0.90                                    # guard jumlah baris
DRY_RUN      = os.environ.get("DRY_RUN", "true").lower() not in ("0", "false", "no")

DETAIL = ["activitydetail", "speed", "rpm", "eta", "remainingdistance", "cargoremarks"]

PROJ_COLS = [
    "id", "code", "remarks", "owner", "started_at", "expected_at",
    "closed_at", "closed_by", "tugboatel.name", "bargeel.name",
    "bargeel.feet", "tugboatel.feet", "customerel.shortname",
    "customerel.fullname", "cargoel.name", "polel.name", "polel.shortname",
    "podel.name", "podel.shortname", "podpreviousel.name", "spal", "si",
    "sidate", "price", "prorata", "bbm", "demmurage", "ondemmurage",
    "sapb1", "projectremarks", "polagen", "podagen",
    "polstartdemmurage", "polenddemmurage",
    "podstartdemmurage", "podenddemmurage", "downtimeamount",
]

HDR = {"apikey": SB_KEY, "Authorization": f"Bearer {SB_KEY}",
       "Content-Type": "application/json"}


# ======================= 1. FETCH =======================
def fetch_records():
    r = requests.get(MERPIS_URL, timeout=180,
                     headers={"Accept": "application/json",
                              "Authorization": MERPIS_TOKEN})
    r.raise_for_status()
    recs = r.json()["data"]["monitoring"]
    n_proj = len({x["project"] for x in recs})
    print(f"Merpis  : {len(recs)} event, {n_proj} project")
    if "year=all" not in MERPIS_URL:
        raise SystemExit("ABORT: MERPIS_API_URL harus memakai year=all")
    return recs


# ======================= 2. BANGUN TABEL TURNAROUND =======================
def build_turnaround(records):
    ev_rows, proj = [], {}
    for r in records:
        p = r.get("projectel") or {}
        a = r.get("activitiesel") or {}
        ev = {k: v for k, v in r.items() if k not in ("projectel", "activitiesel")}
        ev["activity_port"] = a.get("port")
        ev["activity_name"] = a.get("name")
        ev_rows.append(ev)
        if p:
            proj[p["id"]] = p

    df_ev = pd.DataFrame(ev_rows)
    df_proj = pd.json_normalize(list(proj.values()), sep=".")
    df_ev["started_at"] = pd.to_datetime(df_ev["started_at"], errors="coerce")

    # ---- pivot aktivitas per project ----
    ev = df_ev.dropna(subset=["activity_name"]).copy()
    ev["act"] = ev["activity_port"] + " | " + ev["activity_name"]
    ev = ev.sort_values(["project", "started_at", "id"])

    g = ev.groupby(["project", "act"])
    res = (g.agg(Start=("started_at", "first"),
                 Last=("started_at", "last"),
                 Count=("id", "size"))
             .join(g[DETAIL].last())
             .reset_index())

    order = (ev.drop_duplicates("act").sort_values("activities")["act"].tolist())
    parts = []
    for act in order:
        s = res[res["act"] == act].set_index("project")
        b = pd.DataFrame(index=s.index)
        b[f"{act} | Start"] = s["Start"]
        if s["Count"].max() > 1:
            b[f"{act} | Last"] = s["Last"]
            b[f"{act} | Count"] = s["Count"]
        for c in DETAIL:
            if s[c].notna().any():
                b[f"{act} | {c}"] = s[c]
        parts.append(b)
    wide = pd.concat(parts, axis=1)

    # ---- info project ----
    cols = [c for c in PROJ_COLS if c in df_proj.columns]
    info = (df_proj[cols]
            .rename(columns={"id": "project_id", "code": "Project Code"}).copy())

    # ---- T1 = trip sebelumnya (lintas tahun), hanya 2 kolom ----
    pat = info["Project Code"].str.extract(r"^(.*)-(\d{4})-(\d+)$")
    info["_prefix"] = pat[0]
    info["_year"] = pat[1].astype(float)
    info["_trip"] = pat[2].astype(float)
    info = info.sort_values(["_prefix", "_year", "_trip"]).reset_index(drop=True)
    info["T1 Project Code"] = info.groupby("_prefix")["Project Code"].shift(1)
    info["_prev_id"] = info.groupby("_prefix")["project_id"].shift(1)

    t1_src = {
        "T1 FAW Sailing To Finish": "POD | FAW Sailing To Finish | Start",
        "T1 Actual Arrive Finish":  "Finish | Actual Arrive Finish | Start",
    }
    for new, src in t1_src.items():
        info[new] = info["_prev_id"].map(wide[src]) if src in wide.columns else pd.NaT

    df = (info.drop(columns=["_prefix", "_year", "_trip", "_prev_id"])
              .merge(wide.add_prefix("T2 | "), left_on="project_id",
                     right_index=True, how="left")
              .reset_index(drop=True))

    # ---- durasi (hari) ----
    def days(a, b):
        if a not in df.columns or b not in df.columns:
            return pd.Series(np.nan, index=df.index)
        return (df[b] - df[a]).dt.total_seconds() / 86400

    POL_ARR = "T2 | POL | Actual Arrive POL | Start"
    POL_LS  = "T2 | POL | Loading Start | Start"
    POL_LC  = "T2 | POL | Loading Complete | Start"
    POL_FAW = "T2 | POL | FAW Sailing To POD | Start"
    POD_ARR = "T2 | POD | Actual Arrive POD | Start"
    POD_DS  = "T2 | POD | Discharge Start | Start"
    POD_DC  = "T2 | POD | Discharge Complete | Start"
    POD_FAW = "T2 | POD | FAW Sailing To Finish | Start"
    FIN_ARR = "T2 | Finish | Actual Arrive Finish | Start"

    df["Voyage V.2 (hari)"] = days("T1 FAW Sailing To Finish", POD_FAW)
    df["Jeda Finish T1 ke Arrive POL (hari)"] = days("T1 Actual Arrive Finish", POL_ARR)
    df["POL Loading (hari)"] = days(POL_LS, POL_LC)
    df["POL Total Port (hari)"] = days(POL_ARR, POL_FAW)
    df["Sailing POL ke POD (hari)"] = days(POL_FAW, POD_ARR)
    df["POD Discharge (hari)"] = days(POD_DS, POD_DC)
    df["POD Total Port (hari)"] = days(POD_ARR, POD_FAW)
    df["Sailing POD ke Finish (hari)"] = days(POD_FAW, FIN_ARR)
    df["Total Voyage Days"] = (df["Sailing POL ke POD (hari)"]
                               + df["Sailing POD ke Finish (hari)"])

    # ---- flag Data Check (durasi negatif) ----
    labels = {
        "Jeda Finish T1 ke Arrive POL (hari)": "Jeda T1-POL negatif",
        "POL Loading (hari)": "POL Loading negatif",
        "Sailing POL ke POD (hari)": "Sailing POL-POD negatif",
        "Sailing POD ke Finish (hari)": "Sailing POD-Finish negatif",
        "Total Voyage Days": "Total Voyage negatif",
    }
    chk = pd.DataFrame({lab: np.where(df[c] < 0, lab, "") for c, lab in labels.items()})
    df["Data Check"] = (chk.apply(lambda r: "; ".join(x for x in r if x), axis=1)
                           .replace("", None))

    print(f"Turnaround: {df.shape[0]} project x {df.shape[1]} kolom | "
          f"tanpa T1: {df['T1 Project Code'].isna().sum()} | "
          f"flag Data Check: {df['Data Check'].notna().sum()}")
    return df


# ======================= 3. RAPIKAN UNTUK SUPABASE =======================
def snake(name):
    s = name.replace("T2 | ", "t2_").replace("T1 ", "t1_")
    s = s.replace("|", " ").replace("/", " ").replace(".", "_")
    s = re.sub(r"[^0-9a-zA-Z_ ]+", " ", s)
    s = re.sub(r"\s+", "_", s.strip()).lower()
    return re.sub(r"_+", "_", s)

def to_num(s):
    out = pd.to_numeric(s, errors="coerce")
    lost = int((s.notna() & out.isna()).sum())
    if lost:
        print(f"  ! {lost} nilai non-angka di '{s.name}' jadi null")
    return out

def to_dt(s):
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_localize(None)

def finalize(df_turn):
    df = df_turn.copy()
    new = [snake(c) for c in df.columns]
    dup = pd.Series(new)[pd.Series(new).duplicated(keep=False)].tolist()
    assert not dup, f"Nama kolom kembar: {dup}"
    assert max(len(c.encode()) for c in new) <= 63, "Ada nama kolom > 63 karakter"
    df.columns = new

    for c in ["started_at", "expected_at", "closed_at", "sidate"]:
        if c in df.columns:
            df[c] = to_dt(df[c])
    for c in [c for c in df.columns if c.endswith("_eta")]:
        df[c] = to_dt(df[c])

    num = ["price", "demmurage", "bbm", "bargeel_feet", "tugboatel_feet",
           "ondemmurage", "downtimeamount", "polstartdemmurage",
           "polenddemmurage", "podstartdemmurage", "podenddemmurage"]
    for c in df.columns:
        if c in num or c.endswith(("_speed", "_rpm", "_remainingdistance")):
            df[c] = to_num(df[c])
    for c in [c for c in df.columns if c.endswith("_count")]:
        df[c] = to_num(df[c]).astype("Int64")

    dups = df[df[KEY].duplicated(keep=False)][KEY].unique().tolist()
    assert not dups, f"project_code duplikat: {dups[:10]}"
    assert df[KEY].notna().all(), "Ada project_code kosong"

    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = out[c].dt.strftime("%Y-%m-%dT%H:%M:%S")
    out = out.astype(object).where(out.notna(), None)
    return df, out.to_dict(orient="records")


# ======================= 4. SUPABASE =======================
BASE = f"{SB_URL}/rest/v1/{TABLE}"

def sql_type(s):
    if pd.api.types.is_datetime64_any_dtype(s): return "timestamp"
    if pd.api.types.is_bool_dtype(s):           return "boolean"
    if pd.api.types.is_integer_dtype(s):        return "bigint"
    if pd.api.types.is_float_dtype(s):          return "double precision"
    return "text"

def req(method, url, retries=3, **kw):
    extra = kw.pop("headers", {})
    err = ""
    for i in range(retries):
        try:
            r = requests.request(method, url, headers={**HDR, **extra},
                                 timeout=120, **kw)
            if r.status_code < 300:
                return r
            if r.status_code < 500 and r.status_code != 429:
                raise RuntimeError(f"{r.status_code}: {r.text[:500]}")
            err = f"{r.status_code}: {r.text[:200]}"
        except requests.RequestException as e:
            err = str(e)
        time.sleep(2 ** i)
    raise RuntimeError(f"Gagal setelah {retries}x: {err}")

def existing_keys():
    keys, offset, page = [], 0, 1000
    while True:
        rows = req("GET", BASE, params={"select": KEY, "order": KEY,
                                        "limit": page, "offset": offset}).json()
        keys += [x[KEY] for x in rows]
        if len(rows) < page:
            return keys
        offset += page

def check_schema(df):
    """Bandingkan kolom df dengan tabel. Cetak DDL/ALTER lalu stop kalau tidak cocok."""
    try:
        r = requests.get(f"{SB_URL}/rest/v1/", timeout=60,
                         headers={**HDR, "Accept": "application/openapi+json"})
        r.raise_for_status()
        props = r.json().get("definitions", {}).get(TABLE, {}).get("properties")
    except Exception as e:
        print("  ! Cek skema dilewati:", e)
        return
    cols = [c for c in df.columns]
    if not props:
        lines = [f'  "{c}" {"text primary key" if c == KEY else sql_type(df[c])}' for c in cols]
        lines.append('  "synced_at" timestamptz')
        print(f"\nTabel {TABLE} belum ada. Jalankan di SQL Editor:\n")
        print(f"create table {TABLE} (\n" + ",\n".join(lines) + "\n);")
        print("notify pgrst, 'reload schema';")
        raise SystemExit("ABORT: buat tabel dulu")
    missing = [c for c in cols if c not in props]
    if missing:
        print(f"\n{len(missing)} kolom belum ada di tabel. Jalankan di SQL Editor:\n")
        for c in missing:
            print(f'alter table {TABLE} add column if not exists "{c}" {sql_type(df[c])};')
        print("notify pgrst, 'reload schema';")
        raise SystemExit("ABORT: tambah kolom dulu")
    print("Skema   : cocok")

def sync(df, rows):
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    rows = [{**r, "synced_at": now} for r in rows]
    new_keys = [r[KEY] for r in rows]

    check_schema(df)

    old_keys = existing_keys()
    print(f"Supabase: {len(old_keys)} baris | baru: {len(rows)}")
    if not rows:
        raise SystemExit("ABORT: payload kosong")
    if old_keys and len(rows) < MIN_RATIO * len(old_keys):
        raise SystemExit(f"ABORT: jumlah baru turun drastis ({len(rows)} < "
                         f"{MIN_RATIO:.0%} x {len(old_keys)})")

    orphans = sorted(set(old_keys) - set(new_keys))
    print(f"Upsert  : {len(rows)} | dihapus: {len(orphans)} {orphans[:10]}")

    if DRY_RUN:
        print("\nDRY_RUN=true -> tidak ada yang ditulis.")
        return

    dumps = lambda o: json.dumps(o, default=lambda x: x.item() if hasattr(x, "item") else str(x))
    for i in range(0, len(rows), BATCH):
        req("POST", BASE, params={"on_conflict": KEY},
            headers={"Prefer": "resolution=merge-duplicates,return=minimal"},
            data=dumps(rows[i:i + BATCH]))
        print(f"  upsert {min(i + BATCH, len(rows))}/{len(rows)}")

    for i in range(0, len(orphans), 100):
        in_list = ",".join('"' + c.replace('"', "") + '"' for c in orphans[i:i + 100])
        req("DELETE", BASE, params={KEY: f"in.({in_list})"},
            headers={"Prefer": "return=minimal"})
    print(f"  hapus yatim: {len(orphans)}")

    final = existing_keys()
    print(f"\nSelesai. Baris di tabel: {len(final)} (target {len(rows)})")
    assert set(final) == set(new_keys), "Isi tabel tidak sama dengan payload"


# ======================= MAIN =======================
def main():
    records = fetch_records()
    df_turn = build_turnaround(records)
    df, rows = finalize(df_turn)
    sync(df, rows)
    return df_turn          # untuk dicek di Jupyter

if __name__ == "__main__":
    df_turn = main()

Merpis  : 25077 event, 1358 project
Turnaround: 1358 project x 249 kolom | tanpa T1: 43 | flag Data Check: 56
  ! 1 nilai non-angka di 't2_pol_faw_sailing_to_pod_remainingdistance' jadi null
  ! 1 nilai non-angka di 't2_pol_sailing_to_pod_remainingdistance' jadi null
Skema   : cocok
Supabase: 0 baris | baru: 1358
Upsert  : 1358 | dihapus: 0 []

DRY_RUN=true -> tidak ada yang ditulis.


In [4]:
records = fetch_records()   # fetch ulang dari Merpis (data terbaru)

# 1. nilai remainingdistance yang bukan angka
bad = []
for r in records:
    v = r.get("remainingdistance")
    if v not in (None, "") and pd.isna(pd.to_numeric(v, errors="coerce")):
        bad.append({
            "event_id": r["id"],
            "project_code": r["projectel"]["code"],
            "remarks": r["projectel"].get("remarks"),
            "aktivitas": (r.get("activitiesel") or {}).get("name"),
            "started_at": r["started_at"],
            "nilai": repr(v),
            "created_by": r["created_by"],
        })
print("remainingdistance non-angka:")
print(pd.DataFrame(bad).to_string(index=False))

# 2. cek dua project yang sudah dikoreksi
df_turn = build_turnaround(records)
fix = ["BG.MP15-2026-002", "TB.MP39-2026-004", "TB.MP39-2026-005"]
cols = ["Project Code", "T1 Actual Arrive Finish",
        "T2 | POL | Actual Arrive POL | Start",
        "T2 | Finish | Actual Arrive Finish | Start",
        "Jeda Finish T1 ke Arrive POL (hari)",
        "Sailing POD ke Finish (hari)", "Total Voyage Days", "Data Check"]
print("\nProject yang dikoreksi:")
print(df_turn[df_turn["Project Code"].isin(fix)][cols].to_string(index=False))

# 3. sisa flag yang paling besar
x = df_turn[df_turn["Data Check"].notna()]
neg = ["Jeda Finish T1 ke Arrive POL (hari)", "Sailing POD ke Finish (hari)",
       "POL Loading (hari)", "Sailing POL ke POD (hari)", "Total Voyage Days"]
x = x.assign(min_hari=x[neg].min(axis=1)).sort_values("min_hari")
print("\nFlag terbesar (negatif paling dalam):")
print(x[["Project Code", "T1 Project Code", "min_hari", "Data Check"]]
      .head(15).to_string(index=False))

Merpis  : 25094 event, 1359 project
remainingdistance non-angka:
Empty DataFrame
Columns: []
Index: []
Turnaround: 1359 project x 247 kolom | tanpa T1: 43 | flag Data Check: 44

Project yang dikoreksi:
    Project Code T1 Actual Arrive Finish T2 | POL | Actual Arrive POL | Start T2 | Finish | Actual Arrive Finish | Start  Jeda Finish T1 ke Arrive POL (hari)  Sailing POD ke Finish (hari)  Total Voyage Days Data Check
BG.MP15-2026-002     2026-02-02 15:00:00                  2026-02-02 15:00:00                        2026-03-05 15:30:00                                  0.0                      5.812500          11.989583       None
TB.MP39-2026-004     2026-09-15 09:00:00                  2026-09-15 09:00:00                        2026-09-30 15:00:00                                  0.0                      1.916667           4.635417       None
TB.MP39-2026-005     2026-09-30 15:00:00                  2026-09-30 15:00:00                        2026-10-19 16:00:00                        

In [5]:
cek = ["BG.ADT-2024-016", "BG.MP07-2025-006", "BG.PS03-2024-036",
       "BG.ML05-2025-002", "BG.MP07-2025-027", "BG.MP01-2025-028"]
cols = ["Project Code", "T1 Actual Arrive Finish",
        "T2 | POL | Actual Arrive POL | Start",
        "T2 | POL | Loading Start | Start",
        "T2 | POL | Loading Complete | Start",
        "T2 | POL | FAW Sailing To POD | Start",
        "T2 | POD | Actual Arrive POD | Start",
        "T2 | POD | FAW Sailing To Finish | Start",
        "T2 | Finish | Actual Arrive Finish | Start"]
pd.set_option("display.width", 300)
print(df_turn[df_turn["Project Code"].isin(cek)][cols].T.to_string())

                                                           48                   369                  520                  651                  672                  1124
Project Code                                    BG.ADT-2024-016     BG.ML05-2025-002     BG.MP01-2025-028     BG.MP07-2025-006     BG.MP07-2025-027     BG.PS03-2024-036
T1 Actual Arrive Finish                     2024-10-27 04:10:00  2025-02-07 16:30:00  2025-12-27 03:15:00  2025-02-26 21:30:00  2025-12-28 00:30:00  2024-12-24 06:25:00
T2 | POL | Actual Arrive POL | Start        2024-10-27 04:10:00  2025-02-07 16:30:00  2025-12-27 03:15:00  2025-02-26 21:30:00  2025-12-28 00:30:00  2024-12-24 06:25:00
T2 | POL | Loading Start | Start            2024-10-29 17:05:00  2025-02-09 01:10:00  2026-01-02 16:40:00  2025-02-28 16:30:00  2026-01-06 10:45:00  2024-12-30 23:00:00
T2 | POL | Loading Complete | Start         2024-11-01 15:00:00  2025-02-13 08:20:00  2025-12-31 23:59:00  2025-03-03 09:05:00  2025-12-31 23:59:00  2025-0

In [6]:
import os
os.environ["DRY_RUN"] = "false"
%run merpis_turnaround.py

Exception: File `'merpis_turnaround.py'` not found.

In [8]:
%%writefile merpis_turnaround.py
"""
Merpis API (year=all) -> 1 baris per project ...
"""
import os, re, json, time
...
(seluruh isi script, sampai baris terakhir: df_turn = main())

Writing merpis_turnaround.py


In [10]:
%%writefile merpis_turnaround.py
"""
Merpis API (year=all) -> 1 baris per project (T1 ringkas + T2 semua aktivitas)
-> upsert ke Supabase.  Default DRY_RUN=true (tidak menulis apa pun).
"""
import os, re, json, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests

try:
    from dotenv import load_dotenv
    load_dotenv("environment.env", override=True)
except ImportError:
    pass

# ======================= KONFIGURASI =======================
MERPIS_URL   = os.environ["MERPIS_API_URL"]
MERPIS_TOKEN = os.environ["MERPIS_API_TOKEN"]          # sudah berisi "Bearer ..."
SB_URL       = os.environ["SUPABASE_URL"].rstrip("/")
SB_KEY       = os.environ["SUPABASE_KEY"]
TABLE        = os.environ.get("TURNAROUND_TABLE", "monitoring_turnaround_v2")
KEY          = "project_code"
BATCH        = 200
MIN_RATIO    = 0.90                                    # guard jumlah baris
DRY_RUN      = os.environ.get("DRY_RUN", "true").lower() not in ("0", "false", "no")

DETAIL = ["activitydetail", "speed", "rpm", "eta", "remainingdistance", "cargoremarks"]

PROJ_COLS = [
    "id", "code", "remarks", "owner", "started_at", "expected_at",
    "closed_at", "closed_by", "tugboatel.name", "bargeel.name",
    "bargeel.feet", "tugboatel.feet", "customerel.shortname",
    "customerel.fullname", "cargoel.name", "polel.name", "polel.shortname",
    "podel.name", "podel.shortname", "podpreviousel.name", "spal", "si",
    "sidate", "price", "prorata", "bbm", "demmurage", "ondemmurage",
    "sapb1", "projectremarks", "polagen", "podagen",
    "polstartdemmurage", "polenddemmurage",
    "podstartdemmurage", "podenddemmurage", "downtimeamount",
]

HDR = {"apikey": SB_KEY, "Authorization": f"Bearer {SB_KEY}",
       "Content-Type": "application/json"}


# ======================= 1. FETCH =======================
def fetch_records():
    r = requests.get(MERPIS_URL, timeout=180,
                     headers={"Accept": "application/json",
                              "Authorization": MERPIS_TOKEN})
    r.raise_for_status()
    recs = r.json()["data"]["monitoring"]
    n_proj = len({x["project"] for x in recs})
    print(f"Merpis  : {len(recs)} event, {n_proj} project")
    if "year=all" not in MERPIS_URL:
        raise SystemExit("ABORT: MERPIS_API_URL harus memakai year=all")
    return recs


# ======================= 2. BANGUN TABEL TURNAROUND =======================
def build_turnaround(records):
    ev_rows, proj = [], {}
    for r in records:
        p = r.get("projectel") or {}
        a = r.get("activitiesel") or {}
        ev = {k: v for k, v in r.items() if k not in ("projectel", "activitiesel")}
        ev["activity_port"] = a.get("port")
        ev["activity_name"] = a.get("name")
        ev_rows.append(ev)
        if p:
            proj[p["id"]] = p

    df_ev = pd.DataFrame(ev_rows)
    df_proj = pd.json_normalize(list(proj.values()), sep=".")
    df_ev["started_at"] = pd.to_datetime(df_ev["started_at"], errors="coerce")

    # ---- pivot aktivitas per project ----
    ev = df_ev.dropna(subset=["activity_name"]).copy()
    ev["act"] = ev["activity_port"] + " | " + ev["activity_name"]
    ev = ev.sort_values(["project", "started_at", "id"])

    g = ev.groupby(["project", "act"])
    res = (g.agg(Start=("started_at", "first"),
                 Last=("started_at", "last"),
                 Count=("id", "size"))
             .join(g[DETAIL].last())
             .reset_index())

    order = (ev.drop_duplicates("act").sort_values("activities")["act"].tolist())
    parts = []
    for act in order:
        s = res[res["act"] == act].set_index("project")
        b = pd.DataFrame(index=s.index)
        b[f"{act} | Start"] = s["Start"]
        if s["Count"].max() > 1:
            b[f"{act} | Last"] = s["Last"]
            b[f"{act} | Count"] = s["Count"]
        for c in DETAIL:
            if s[c].notna().any():
                b[f"{act} | {c}"] = s[c]
        parts.append(b)
    wide = pd.concat(parts, axis=1)

    # ---- info project ----
    cols = [c for c in PROJ_COLS if c in df_proj.columns]
    info = (df_proj[cols]
            .rename(columns={"id": "project_id", "code": "Project Code"}).copy())

    # ---- T1 = trip sebelumnya (lintas tahun), hanya 2 kolom ----
    pat = info["Project Code"].str.extract(r"^(.*)-(\d{4})-(\d+)$")
    info["_prefix"] = pat[0]
    info["_year"] = pat[1].astype(float)
    info["_trip"] = pat[2].astype(float)
    info = info.sort_values(["_prefix", "_year", "_trip"]).reset_index(drop=True)
    info["T1 Project Code"] = info.groupby("_prefix")["Project Code"].shift(1)
    info["_prev_id"] = info.groupby("_prefix")["project_id"].shift(1)

    t1_src = {
        "T1 FAW Sailing To Finish": "POD | FAW Sailing To Finish | Start",
        "T1 Actual Arrive Finish":  "Finish | Actual Arrive Finish | Start",
    }
    for new, src in t1_src.items():
        info[new] = info["_prev_id"].map(wide[src]) if src in wide.columns else pd.NaT

    df = (info.drop(columns=["_prefix", "_year", "_trip", "_prev_id"])
              .merge(wide.add_prefix("T2 | "), left_on="project_id",
                     right_index=True, how="left")
              .reset_index(drop=True))

    # ---- durasi (hari) ----
    def days(a, b):
        if a not in df.columns or b not in df.columns:
            return pd.Series(np.nan, index=df.index)
        return (df[b] - df[a]).dt.total_seconds() / 86400

    POL_ARR = "T2 | POL | Actual Arrive POL | Start"
    POL_LS  = "T2 | POL | Loading Start | Start"
    POL_LC  = "T2 | POL | Loading Complete | Start"
    POL_FAW = "T2 | POL | FAW Sailing To POD | Start"
    POD_ARR = "T2 | POD | Actual Arrive POD | Start"
    POD_DS  = "T2 | POD | Discharge Start | Start"
    POD_DC  = "T2 | POD | Discharge Complete | Start"
    POD_FAW = "T2 | POD | FAW Sailing To Finish | Start"
    FIN_ARR = "T2 | Finish | Actual Arrive Finish | Start"

    df["Voyage V.2 (hari)"] = days("T1 FAW Sailing To Finish", POD_FAW)
    df["Jeda Finish T1 ke Arrive POL (hari)"] = days("T1 Actual Arrive Finish", POL_ARR)
    df["POL Loading (hari)"] = days(POL_LS, POL_LC)
    df["POL Total Port (hari)"] = days(POL_ARR, POL_FAW)
    df["Sailing POL ke POD (hari)"] = days(POL_FAW, POD_ARR)
    df["POD Discharge (hari)"] = days(POD_DS, POD_DC)
    df["POD Total Port (hari)"] = days(POD_ARR, POD_FAW)
    df["Sailing POD ke Finish (hari)"] = days(POD_FAW, FIN_ARR)
    df["Total Voyage Days"] = (df["Sailing POL ke POD (hari)"]
                               + df["Sailing POD ke Finish (hari)"])

    # ---- flag Data Check (durasi negatif) ----
    labels = {
        "Jeda Finish T1 ke Arrive POL (hari)": "Jeda T1-POL negatif",
        "POL Loading (hari)": "POL Loading negatif",
        "Sailing POL ke POD (hari)": "Sailing POL-POD negatif",
        "Sailing POD ke Finish (hari)": "Sailing POD-Finish negatif",
        "Total Voyage Days": "Total Voyage negatif",
    }
    chk = pd.DataFrame({lab: np.where(df[c] < 0, lab, "") for c, lab in labels.items()})
    df["Data Check"] = (chk.apply(lambda r: "; ".join(x for x in r if x), axis=1)
                           .replace("", None))

    print(f"Turnaround: {df.shape[0]} project x {df.shape[1]} kolom | "
          f"tanpa T1: {df['T1 Project Code'].isna().sum()} | "
          f"flag Data Check: {df['Data Check'].notna().sum()}")
    return df


# ======================= 3. RAPIKAN UNTUK SUPABASE =======================
def snake(name):
    s = name.replace("T2 | ", "t2_").replace("T1 ", "t1_")
    s = s.replace("|", " ").replace("/", " ").replace(".", "_")
    s = re.sub(r"[^0-9a-zA-Z_ ]+", " ", s)
    s = re.sub(r"\s+", "_", s.strip()).lower()
    return re.sub(r"_+", "_", s)

def to_num(s):
    out = pd.to_numeric(s, errors="coerce")
    lost = int((s.notna() & out.isna()).sum())
    if lost:
        print(f"  ! {lost} nilai non-angka di '{s.name}' jadi null")
    return out

def to_dt(s):
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_localize(None)

def finalize(df_turn):
    df = df_turn.copy()
    new = [snake(c) for c in df.columns]
    dup = pd.Series(new)[pd.Series(new).duplicated(keep=False)].tolist()
    assert not dup, f"Nama kolom kembar: {dup}"
    assert max(len(c.encode()) for c in new) <= 63, "Ada nama kolom > 63 karakter"
    df.columns = new

    for c in ["started_at", "expected_at", "closed_at", "sidate"]:
        if c in df.columns:
            df[c] = to_dt(df[c])
    for c in [c for c in df.columns if c.endswith("_eta")]:
        df[c] = to_dt(df[c])

    num = ["price", "demmurage", "bbm", "bargeel_feet", "tugboatel_feet",
           "ondemmurage", "downtimeamount", "polstartdemmurage",
           "polenddemmurage", "podstartdemmurage", "podenddemmurage"]
    for c in df.columns:
        if c in num or c.endswith(("_speed", "_rpm", "_remainingdistance")):
            df[c] = to_num(df[c])
    for c in [c for c in df.columns if c.endswith("_count")]:
        df[c] = to_num(df[c]).astype("Int64")

    dups = df[df[KEY].duplicated(keep=False)][KEY].unique().tolist()
    assert not dups, f"project_code duplikat: {dups[:10]}"
    assert df[KEY].notna().all(), "Ada project_code kosong"

    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = out[c].dt.strftime("%Y-%m-%dT%H:%M:%S")
    out = out.astype(object).where(out.notna(), None)
    return df, out.to_dict(orient="records")


# ======================= 4. SUPABASE =======================
BASE = f"{SB_URL}/rest/v1/{TABLE}"

def sql_type(s):
    if pd.api.types.is_datetime64_any_dtype(s): return "timestamp"
    if pd.api.types.is_bool_dtype(s):           return "boolean"
    if pd.api.types.is_integer_dtype(s):        return "bigint"
    if pd.api.types.is_float_dtype(s):          return "double precision"
    return "text"

def req(method, url, retries=3, **kw):
    extra = kw.pop("headers", {})
    err = ""
    for i in range(retries):
        try:
            r = requests.request(method, url, headers={**HDR, **extra},
                                 timeout=120, **kw)
            if r.status_code < 300:
                return r
            if r.status_code < 500 and r.status_code != 429:
                raise RuntimeError(f"{r.status_code}: {r.text[:500]}")
            err = f"{r.status_code}: {r.text[:200]}"
        except requests.RequestException as e:
            err = str(e)
        time.sleep(2 ** i)
    raise RuntimeError(f"Gagal setelah {retries}x: {err}")

def existing_keys():
    keys, offset, page = [], 0, 1000
    while True:
        rows = req("GET", BASE, params={"select": KEY, "order": KEY,
                                        "limit": page, "offset": offset}).json()
        keys += [x[KEY] for x in rows]
        if len(rows) < page:
            return keys
        offset += page

def check_schema(df):
    """Bandingkan kolom df dengan tabel. Cetak DDL/ALTER lalu stop kalau tidak cocok."""
    try:
        r = requests.get(f"{SB_URL}/rest/v1/", timeout=60,
                         headers={**HDR, "Accept": "application/openapi+json"})
        r.raise_for_status()
        props = r.json().get("definitions", {}).get(TABLE, {}).get("properties")
    except Exception as e:
        print("  ! Cek skema dilewati:", e)
        return
    cols = [c for c in df.columns]
    if not props:
        lines = [f'  "{c}" {"text primary key" if c == KEY else sql_type(df[c])}' for c in cols]
        lines.append('  "synced_at" timestamptz')
        print(f"\nTabel {TABLE} belum ada. Jalankan di SQL Editor:\n")
        print(f"create table {TABLE} (\n" + ",\n".join(lines) + "\n);")
        print("notify pgrst, 'reload schema';")
        raise SystemExit("ABORT: buat tabel dulu")
    missing = [c for c in cols if c not in props]
    if missing:
        print(f"\n{len(missing)} kolom belum ada di tabel. Jalankan di SQL Editor:\n")
        for c in missing:
            print(f'alter table {TABLE} add column if not exists "{c}" {sql_type(df[c])};')
        print("notify pgrst, 'reload schema';")
        raise SystemExit("ABORT: tambah kolom dulu")
    print("Skema   : cocok")

def sync(df, rows):
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    rows = [{**r, "synced_at": now} for r in rows]
    new_keys = [r[KEY] for r in rows]

    check_schema(df)

    old_keys = existing_keys()
    print(f"Supabase: {len(old_keys)} baris | baru: {len(rows)}")
    if not rows:
        raise SystemExit("ABORT: payload kosong")
    if old_keys and len(rows) < MIN_RATIO * len(old_keys):
        raise SystemExit(f"ABORT: jumlah baru turun drastis ({len(rows)} < "
                         f"{MIN_RATIO:.0%} x {len(old_keys)})")

    orphans = sorted(set(old_keys) - set(new_keys))
    print(f"Upsert  : {len(rows)} | dihapus: {len(orphans)} {orphans[:10]}")

    if DRY_RUN:
        print("\nDRY_RUN=true -> tidak ada yang ditulis.")
        return

    dumps = lambda o: json.dumps(o, default=lambda x: x.item() if hasattr(x, "item") else str(x))
    for i in range(0, len(rows), BATCH):
        req("POST", BASE, params={"on_conflict": KEY},
            headers={"Prefer": "resolution=merge-duplicates,return=minimal"},
            data=dumps(rows[i:i + BATCH]))
        print(f"  upsert {min(i + BATCH, len(rows))}/{len(rows)}")

    for i in range(0, len(orphans), 100):
        in_list = ",".join('"' + c.replace('"', "") + '"' for c in orphans[i:i + 100])
        req("DELETE", BASE, params={KEY: f"in.({in_list})"},
            headers={"Prefer": "return=minimal"})
    print(f"  hapus yatim: {len(orphans)}")

    final = existing_keys()
    print(f"\nSelesai. Baris di tabel: {len(final)} (target {len(rows)})")
    assert set(final) == set(new_keys), "Isi tabel tidak sama dengan payload"


# ======================= MAIN =======================
def main():
    records = fetch_records()
    df_turn = build_turnaround(records)
    df, rows = finalize(df_turn)
    sync(df, rows)
    return df_turn          # untuk dicek di Jupyter

if __name__ == "__main__":
    df_turn = main()

Overwriting merpis_turnaround.py


In [12]:
import os
os.environ["DRY_RUN"] = "false"     # dry run
%run merpis_turnaround.py

Merpis  : 25126 event, 1359 project
Turnaround: 1359 project x 247 kolom | tanpa T1: 43 | flag Data Check: 44
Skema   : cocok
Supabase: 0 baris | baru: 1359
Upsert  : 1359 | dihapus: 0 []
  upsert 200/1359
  upsert 400/1359
  upsert 600/1359
  upsert 800/1359
  upsert 1000/1359
  upsert 1200/1359
  upsert 1359/1359
  hapus yatim: 0

Selesai. Baris di tabel: 1359 (target 1359)


In [14]:
import os

print(os.getcwd())
print([f for f in os.listdir() if f.endswith((".py", ".env"))])

os.environ["V2_DRY_RUN"] = "true"
%run main.py

C:\Users\fahru\Merpis_Pipeline\Detailing_Monitoring
['environment.env', 'merpis_turnaround.py']


Exception: File `'main.py'` not found.